In [1]:
import struct
import numpy as np
import math
from numpy.random import *
import WinoTran_CHWN as CHWN

In [2]:
#generate random parameters
M = 1280
N = 3200
K = 128
Batch = 128
parameters1 = M * K * Batch
parameters2 = N * K * Batch

input1 = (np.array(rand(parameters1))-0.5).astype(np.float32)
des = open("input.bin","wb")
cnt = des.write(input1)
des.close()

kernel = (np.array(rand(parameters2))-0.5).astype(np.float32)
des = open("filter.bin","wb")
cnt = des.write(kernel)
des.close()

In [3]:
# Top layer of design
# control the data flow and the parameters.

# verify conv result
# chwn
inside = 22
numOfFilter = 128
padding = 1
chn = 128
bat4Conv = 64
oside = inside + 2*padding - 2


parameters1 = bat4Conv * inside * inside * chn
parameters2 = numOfFilter * 9 * chn 

inside_beta = math.ceil((inside+2*padding-2)/4)*4+2 
blockn = (int)((inside_beta-2)/4)

M = (int)(blockn * blockn * bat4Conv);
MSize = M if (M%128 == 0) else math.ceil(M/128)*128

N = numOfFilter
NSize = (int((N-1)/128)+1)*128
K = chn
KSize = (int((K-1)/8)+1)*8
kernelTran = np.zeros((36,KSize,NSize)).astype(np.float32)


# readin the feature map
src = open("input.bin","rb")
context = src.read(parameters1*4)
real_context = struct.unpack(str(parameters1)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
sample_input = input.reshape((chn,inside,inside,bat4Conv)).astype(np.float32)


# readin the kernel map
src = open("filter.bin","rb")
context = src.read(parameters2*4)
real_context = struct.unpack(str(parameters2)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
sample_kernel = input.reshape((chn,3,3,numOfFilter)).astype(np.float32)


# sample_input = np.ones((chn,inside,inside,bat4Conv)).astype(np.float32)
# sample_kernel = np.ones((chn,3,3,numOfFilter)).astype(np.float32)

DEBUG =1
if (DEBUG):
    print("inside:",inside,"  inside_beta:",inside_beta)
    print("Blockn:",blockn)
    print("M:",M,"  N:",N,"  K:",K)
    print("MSize:",MSize,"  NSize:",NSize,"  KSize:",KSize)
    print("input_Shape:",sample_input.shape,"   Kernel_Shape:", sample_kernel.shape)
    
#     print(kernelTran.shape)
    

# inputTran2 = Wino_inputTran2(sample_input)
inputTran = CHWN.Wino_inputTran(sample_input,padding)
kernelTran = CHWN.Wino_kernelTran(sample_kernel)

if (DEBUG):
    print()
    print("inputTran_Shape:",inputTran.shape)
    print("kernelTran_Shape:",kernelTran.shape)
    print()
    
gemmResult = CHWN.gemm(inputTran,kernelTran,MSize,NSize,DEBUG)

if (DEBUG):
    print()
    print("gemmResult_Shape:",gemmResult.shape)

gemmTran = CHWN.Wino_OutputTran(gemmResult,oside,bat4Conv,numOfFilter)
finalOutput = CHWN.Wino_inverseTran(gemmTran, numOfFilter, bat4Conv, blockn, oside)
print(finalOutput.shape)

inside: 22   inside_beta: 26
Blockn: 6
M: 2304   N: 128   K: 128
MSize: 2304   NSize: 128   KSize: 128
input_Shape: (128, 22, 22, 64)    Kernel_Shape: (128, 3, 3, 128)

inputTran_Shape: (36, 128, 2304)
kernelTran_Shape: (36, 128, 128)


gemmResult_Shape: (36, 2304, 128)
(128, 22, 22, 64)


In [19]:
print(bat4Conv)
numOfFilter = chn if (chn%128 == 0) else math.ceil(chn/128)*128
assert(numOfFilter == gemmTran.shape[1])
assert(bat4Conv == gemmTran.shape[0])
assert(blockn == ( gemmTran.shape[2]/6) )

oside_beta = blockn *4

output = np.zeros((chn,oside_beta,oside_beta,bat4Conv)).astype(np.float32)
finalOutput = np.zeros((chn,oside,oside,bat4Conv)).astype(np.float32)
print(finalOutput.shape)

64
(128, 22, 22, 64)


In [4]:
testoutput = CHWN.Conv_CHWN(sample_input, sample_kernel,padding)
print(testoutput.shape)
print(finalOutput.shape)
assert(testoutput.shape == finalOutput.shape)

err = np.sum(np.abs(testoutput - finalOutput))
print(err)

(128, 22, 22, 64)
(128, 22, 22, 64)
13.375903


**Above Code is used for CHWN kernel test**

In [23]:
nInputTran = 36*MSize*KSize
# print(MSize, KSize)

#inputTran
src = open("output1.bin","rb")
context = src.read( )
# print(nInputTran)
# print(len(context)/4)
# print(len(context)/4)
# print(nInputTran)
assert(len(context) == nInputTran*4)
real_context = struct.unpack(str(nInputTran)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
cuda_InputTran = input.reshape((36,KSize,MSize)).astype(np.float32)


In [24]:
print(cuda_InputTran[0,:5,:5])
print(inputTran[0,:5,:5])


[[ -3.158599   -5.556092    2.64573    -5.1685457  11.238125 ]
 [  7.2410793   5.29173    -4.919345    8.885408   -5.246728 ]
 [ -9.700469   -8.40131   -11.115476   11.314179    3.0462353]
 [ -4.53967    -3.4403749   9.061494   -7.3775377  -3.460187 ]
 [ -1.8297265   5.173499    4.167367  -12.008635    6.2270103]]
[[ -3.158599   -5.5560923   2.64573    -5.1685457  11.238124 ]
 [  7.2410803   5.2917295  -4.919345    8.885408   -5.2467275]
 [ -9.70047    -8.401311  -11.115477   11.314179    3.0462353]
 [ -4.5396695  -3.4403746   9.061493   -7.3775373  -3.460187 ]
 [ -1.8297265   5.173499    4.167367  -12.008635    6.2270103]]


In [18]:
nInputTran = 36*MSize*KSize
nKernelTran = 36*NSize*KSize
nGemmOutput = 36*MSize*NSize
nConvOutput = bat4Conv*oside*oside*numOfFilter

#inputTran
src = open("Module1.bin","rb")
context = src.read( )
# print(nInputTran)
# print(len(context)/4)
assert(len(context) == nInputTran*4)
real_context = struct.unpack(str(nInputTran)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
cuda_InputTran = input.reshape((36,KSize,MSize)).astype(np.float32)


#kernelTran
src = open("Module2.bin","rb")
context = src.read()
assert(len(context) == nKernelTran*4)
real_context = struct.unpack(str(nKernelTran)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
cuda_FilterTran = input.reshape((36,KSize,NSize)).astype(np.float32)


#GEMMOutput
src = open("Module3.bin","rb")
context = src.read()
# print(len(context)/4)
# print(nGemmOutput)
assert(len(context) == nGemmOutput*4)
real_context = struct.unpack(str(nGemmOutput)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
cuda_GemmOutput = input.reshape((36,MSize,NSize)).astype(np.float32)


#GEMMOutput
src = open("Asura.bin","rb")
context = src.read()
assert(len(context) == nConvOutput*4)
real_context = struct.unpack(str(nConvOutput)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
cuda_ConvOutput = input.reshape((bat4Conv,numOfFilter,oside,oside)).astype(np.float32)

In [19]:
# for inputTran

# print(cuda_InputTran.shape)
# print(inputTran.shape)
assert(cuda_InputTran.shape == inputTran.shape)
assert(cuda_FilterTran.shape == kernelTran.shape)
assert(cuda_GemmOutput.shape == gemmResult.shape)
assert(cuda_ConvOutput.shape == finalOutput.shape)

# err0 = np.sum(np.abs(cuda_InputTran - inputTran2))
err1 = np.sum(np.abs(cuda_InputTran - inputTran))
err2 = np.sum(np.abs(cuda_FilterTran - kernelTran))
# print(err0)
print(err1,err2)

err3 = np.sum(np.abs(cuda_GemmOutput - gemmResult))
print(err3)

err4 = np.sum(np.abs(cuda_ConvOutput - finalOutput))
print(err4)
# diff = finalOutput - cuda_ConvOutput

2.2351296 0.00046742253
3.5836694
9.87331


In [9]:
nConvOutput = bat4Conv*(oside)*(oside)*numOfFilter
#GEMMOutput
src = open("ConvModule_NCHW.bin","rb")
context = src.read()
print(nConvOutput, len(context))

assert(len(context) == nConvOutput*4)
real_context = struct.unpack(str(nConvOutput)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
cuda_ConvModule = input.reshape((bat4Conv,numOfFilter,oside,oside)).astype(np.float32)

3964928 15859712


In [10]:
assert(cuda_ConvModule.shape == finalOutput.shape)
err = np.sum(np.abs(cuda_ConvModule - finalOutput))
print(err)

12.886401


In [29]:
print(cuda_ConvModule[0,0,:5,:5])
print(finalOutput[0,0,:5,:5])

[[ 0.53605086 -4.3838997  -5.1545353  -0.15999937 -2.5982037 ]
 [ 0.902688    2.2523973   1.9864316  -3.0306797  -1.4030911 ]
 [ 0.19874859 -2.1245184   1.78311    -0.24964428  1.908009  ]
 [ 0.5811477   7.313672    3.1764297   2.6240273   1.4857343 ]
 [-0.786922   -3.0025396   1.3739996   6.295079   -0.1813246 ]]
[[ 0.53604984 -4.383898   -5.1545362  -0.1599965  -2.5982049 ]
 [ 0.9026875   2.2523968   1.9864316  -3.0306888  -1.4030887 ]
 [ 0.19874913 -2.1245165   1.783109   -0.24963965  1.9080081 ]
 [ 0.58114743  7.313676    3.1764367   2.6240232   1.4857352 ]
 [-0.7869217  -3.002542    1.3740014   6.2950673  -0.18132457]]


In [12]:
print(cuda_ConvModule[0,0,:5,:5])
print(finalOutput[0,0,:5,:5])

[[-3.1312537  -5.193768   -0.5433121  -1.1102772   5.663509  ]
 [ 0.7761052  -2.0598507   3.238768   -2.4961014  -1.5177099 ]
 [-5.3421574  -6.095705    4.498821   -0.23359299  1.4550455 ]
 [ 0.9372153  -2.0244455  -2.125268    3.711238   -4.6974382 ]
 [-3.602247   -2.5058064  -1.1003078   0.50709033  2.2523947 ]]
[[ 0.53604984 -4.383898   -5.1545362  -0.1599965  -2.5982049 ]
 [ 0.9026875   2.2523968   1.9864316  -3.0306888  -1.4030887 ]
 [ 0.19874913 -2.1245165   1.783109   -0.24963965  1.9080081 ]
 [ 0.58114743  7.313676    3.1764367   2.6240232   1.4857352 ]
 [-0.7869217  -3.002542    1.3740014   6.2950673  -0.18132457]]


In [8]:
nConvOutput = bat4Conv*oside*oside*numOfFilter
#GEMMOutput
src = open("Cu_output.bin","rb")
context = src.read()
assert(len(context) == nConvOutput*4)
real_context = struct.unpack(str(nConvOutput)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
cudnn_ConvModule = input.reshape((bat4Conv,numOfFilter,oside,oside)).astype(np.float32)

In [4]:
nConvOutput = bat4Conv*oside*oside*numOfFilter
#GEMMOutput
src = open("Cu_output2.bin","rb")
context = src.read()
assert(len(context) == nConvOutput*4)
real_context = struct.unpack(str(nConvOutput)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
cudnn_ConvModule2 = input.reshape((bat4Conv,numOfFilter,oside,oside)).astype(np.float32)

In [5]:
assert(cudnn_ConvModule2.shape == finalOutput.shape)
err = np.sum(np.abs(cudnn_ConvModule2 - finalOutput))
print(err)

23.044996


In [11]:
assert(cudnn_ConvModule.shape == finalOutput.shape)
err = np.sum(np.abs(cudnn_ConvModule - finalOutput))
print(err)

19.974483


In [12]:
assert(cuda_ConvModule.shape == cudnn_ConvModule.shape)
err = np.sum(np.abs(cuda_ConvModule - cudnn_ConvModule))
print(err)

20.290037


In [9]:
import sys
print(sys.executable)

/home/elon/Desktop/Workspace/env/bin/python
